# Solution 4 — Fine-tuning a dialect-robust retrieval encoder

Every mitigation so far patches the **query** after the fact, or swaps in a different off-the-shelf encoder. None recovered a significant amount across 4 encoders × 4 mitigations × 2 subsets.

The gap lives in the **representation** — the encoder puts the MSA and Darija forms of the same question in different places. This trains it not to, using your 500 aligned pairs with `MultipleNegativesRankingLoss`.

**Critical methodology:** the split is by **source passage**, not by question. Two questions grounded in the same passage can't straddle train/test, or the model memorises passage vocabulary and the test score is inflated. Test questions *and* their gold passages are held out entirely.

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **GPU runtime required.** No API key.

### Install

In [ ]:
# !pip install -q faiss-cpu rank_bm25 sentence-transformers requests accelerate

### Config

In [ ]:
CONFIG = {
    "corpus_file": "corpus_v2.json",
    "wiki_qa_file": "qa_pairs_wiki.json",
    "repo_raw": "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main",

    "base_encoder": "intfloat/multilingual-e5-base",   # best from Solution 3
    "alpha": 0.8,                                      # its best hybrid weight

    "test_frac": 0.35,        # held-out share of passages
    "epochs": 3,
    "batch_size": 16,
    "lr": 2e-5,
    "warmup_frac": 0.1,
    "seed": 42,

    "k_values": (1, 3, 5, 10),
    "max_k": 10,
    "bootstrap_n": 1000,
    "ci": 95,
}

### Load data

In [ ]:
import json, requests, random
import numpy as np

with open(CONFIG["corpus_file"], encoding="utf-8") as f:
    corpus = json.load(f)
with open(CONFIG["wiki_qa_file"], encoding="utf-8") as f:
    wiki_qa = json.load(f)
pilot_qa = json.loads(requests.get(f"{CONFIG['repo_raw']}/data/qa_pairs.json").text)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)

all_qa = [q for q in (wiki_qa + pilot_qa) if q["source_chunk_id"] in known]
print(f"Corpus: {len(corpus)} passages | usable QA pairs: {len(all_qa)}")

### Passage-level split (prevents leakage)

In [ ]:
random.seed(CONFIG["seed"])

gold_passages = sorted({q["source_chunk_id"] for q in all_qa})
random.shuffle(gold_passages)
n_test = int(len(gold_passages) * CONFIG["test_frac"])
test_passages = set(gold_passages[:n_test])
train_passages = set(gold_passages[n_test:])

train_qa = [q for q in all_qa if q["source_chunk_id"] in train_passages]
test_qa  = [q for q in all_qa if q["source_chunk_id"] in test_passages]

# Sanity: no passage may appear on both sides.
assert not (train_passages & test_passages), "passage leakage between splits"

def subset_of(q):
    return "wikipedia" if q["source_chunk_id"].startswith("wiki_") else "pilot"

print(f"Gold passages: {len(gold_passages)}  ->  train {len(train_passages)} / test {len(test_passages)}")
print(f"Questions:     train {len(train_qa)} / test {len(test_qa)}")
from collections import Counter
print("Test composition:", dict(Counter(subset_of(q) for q in test_qa)))

### Arabic normalization + BM25 (unchanged from earlier notebooks)

In [ ]:
import re
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t)
    t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t)
    t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
print("BM25 index built.")

### Build training pairs

In [ ]:
# Two complementary objectives, both drawn only from TRAIN passages:
#   (a) darija_query  <-> gold passage   teaches dialect-to-document retrieval
#   (b) darija_query  <-> msa_query      pulls the two surface forms of the
#                                         same question together in embedding space
from sentence_transformers import InputExample

train_examples = []
for q in train_qa:
    passage = corpus_map[q["source_chunk_id"]]
    train_examples.append(InputExample(texts=[f'query: {q["darija_query"]}',
                                              f'passage: {passage}']))
    train_examples.append(InputExample(texts=[f'query: {q["darija_query"]}',
                                              f'query: {q["msa_query"]}']))

random.shuffle(train_examples)
print(f"Training examples: {len(train_examples)}")
print("  (a) darija -> gold passage")
print("  (b) darija -> msa paraphrase")

### Evaluation machinery (shared by before/after)

In [ ]:
def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def build_index(model):
    emb = model.encode([f"passage: {t}" for t in corpus_texts],
                       normalize_embeddings=True, show_progress_bar=True, batch_size=32)
    return np.asarray(emb, "float32")

def retrieve(model, corpus_emb, query, k, alpha):
    s = 0.0
    if alpha > 0:
        q = model.encode([f"query: {query}"], normalize_embeddings=True)[0]
        s = alpha * minmax(corpus_emb @ q)
    if alpha < 1:
        s = s + (1 - alpha) * minmax(np.asarray(bm25.get_scores(tokenize(query))))
    return [corpus_ids[i] for i in np.argsort(-s)[:k]]

def per_item(model, corpus_emb, items, field, alpha):
    hits = {k: [] for k in CONFIG["k_values"]}
    rr = []
    for it in items:
        got = retrieve(model, corpus_emb, it[field], CONFIG["max_k"], alpha)
        gold = it["source_chunk_id"]
        for k in CONFIG["k_values"]:
            hits[k].append(1.0 if gold in got[:k] else 0.0)
        rr.append(1.0 / (got.index(gold) + 1) if gold in got else 0.0)
    return {**{f"R@{k}": np.array(v) for k, v in hits.items()}, "MRR": np.array(rr)}

### BASELINE: evaluate the off-the-shelf encoder on the held-out test set

In [ ]:
from sentence_transformers import SentenceTransformer
import torch, gc

print("Evaluating base encoder on held-out test questions...")
base_model = SentenceTransformer(CONFIG["base_encoder"])
base_emb = build_index(base_model)

before = {}
for field in ["msa_query", "darija_query"]:
    before[field] = per_item(base_model, base_emb, test_qa, field, CONFIG["alpha"])
    m = before[field]
    print(f"  {field:<14} R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}  MRR={m['MRR'].mean():.3f}")

del base_model, base_emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Fine-tune

In [ ]:
from sentence_transformers import losses
from torch.utils.data import DataLoader

model = SentenceTransformer(CONFIG["base_encoder"])
loader = DataLoader(train_examples, shuffle=True, batch_size=CONFIG["batch_size"])
# MultipleNegativesRankingLoss uses the other items in the batch as negatives,
# which suits paired data where explicit negatives are not available.
loss = losses.MultipleNegativesRankingLoss(model)

steps = len(loader) * CONFIG["epochs"]
model.fit(
    train_objectives=[(loader, loss)],
    epochs=CONFIG["epochs"],
    warmup_steps=int(steps * CONFIG["warmup_frac"]),
    optimizer_params={"lr": CONFIG["lr"]},
    show_progress_bar=True,
)
model.save("finetuned-dialect-encoder")
print("Fine-tuning complete; model saved to ./finetuned-dialect-encoder")

### AFTER: evaluate the fine-tuned encoder on the same held-out set

In [ ]:
print("Re-indexing corpus with the fine-tuned encoder...")
ft_emb = build_index(model)

after = {}
for field in ["msa_query", "darija_query"]:
    after[field] = per_item(model, ft_emb, test_qa, field, CONFIG["alpha"])
    m = after[field]
    print(f"  {field:<14} R@1={m['R@1'].mean():.3f}  R@5={m['R@5'].mean():.3f}  MRR={m['MRR'].mean():.3f}")

### Did fine-tuning close the gap? (paired bootstrap CIs)

In [ ]:
import pandas as pd

rng = np.random.default_rng(CONFIG["seed"])

def boot_diff(a, b):
    d = a - b
    idx = rng.integers(0, len(d), size=(CONFIG["bootstrap_n"], len(d)))
    means = d[idx].mean(axis=1)
    lo, hi = np.percentile(means, [(100 - CONFIG["ci"]) / 2, 100 - (100 - CONFIG["ci"]) / 2])
    return d.mean(), lo, hi

rows = []
for metric in ["R@1", "R@5", "MRR"]:
    # dialect gap, before and after
    gb, gb_lo, gb_hi = boot_diff(before["msa_query"][metric], before["darija_query"][metric])
    ga, ga_lo, ga_hi = boot_diff(after["msa_query"][metric],  after["darija_query"][metric])
    # improvement on dialectal queries specifically
    di, di_lo, di_hi = boot_diff(after["darija_query"][metric], before["darija_query"][metric])
    rows.append({
        "metric": metric,
        "gap_before": gb, "gap_after": ga,
        "gap_after_lo": ga_lo, "gap_after_hi": ga_hi,
        "gap_closed": gb - ga,
        "darija_gain": di, "gain_lo": di_lo, "gain_hi": di_hi,
        "gain_significant": "yes" if di_lo > 0 else "no",
        "gap_eliminated": "yes" if ga_lo <= 0 else "no",
    })

res = pd.DataFrame(rows)
res.to_csv("finetune_results.csv", index=False)

print("=" * 92)
print("FINE-TUNING RESULT — held-out test questions, unseen gold passages")
print("=" * 92)
print(res.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print()
print("gain_significant = yes  -> fine-tuning improved dialectal retrieval (CI excludes 0)")
print("gap_eliminated   = yes  -> the remaining MSA/Darija gap is no longer significant")

### Breakdown by subset, and a sanity check on MSA regression

In [ ]:
print("\n=== Per-subset test results ===\n")
sub_rows = []
for name in ["wikipedia", "pilot"]:
    idx = [i for i, q in enumerate(test_qa) if subset_of(q) == name]
    if not idx:
        continue
    idx = np.array(idx)
    for field in ["msa_query", "darija_query"]:
        for stage, store in [("before", before), ("after", after)]:
            sub_rows.append({
                "subset": name, "n": len(idx), "query": field, "stage": stage,
                "R@1": store[field]["R@1"][idx].mean(),
                "R@5": store[field]["R@5"][idx].mean(),
                "MRR": store[field]["MRR"][idx].mean(),
            })
subs = pd.DataFrame(sub_rows)
print(subs.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
subs.to_csv("finetune_by_subset.csv", index=False)

msa_change = after["msa_query"]["R@5"].mean() - before["msa_query"]["R@5"].mean()
print(f"\nMSA Recall@5 change after fine-tuning: {msa_change:+.3f}")
print("A large negative value would mean the model traded MSA performance for")
print("dialect performance rather than genuinely closing the gap — worth checking.")

### Download

In [ ]:
from google.colab import files
files.download("finetune_results.csv")
files.download("finetune_by_subset.csv")